In [5]:
import os
import pandas as pd
from nilearn import image, plotting, input_data
import matplotlib.pyplot as plt
from bids import BIDSLayout

# Your predefined groups from your research
mld_subs = ['059', '065', '067', '069', '071', '075', '076', '077', '078', '083', '088', '095', '096', '103', '106']
td_subs = ['090', '036', '013', '008', '057', '070', '023', '024', '053', '044', '034', '060', '007', '027', '010']
all_subs = mld_subs + td_subs

data_path = '/Volumes/T9/ds001486/derivatives/fmriprep'
layout = BIDSLayout(data_path, validate=False, config=['bids', 'derivatives'])

# DEBUG: Check if layout found anything at all
all_bolds = layout.get(task='Mult', suffix='bold', desc='preproc')
print(f"Total BOLD files found: {len(all_bolds)}")

output_qc_dir = os.path.expanduser('~/Desktop/mld_qc_plots')
os.makedirs(output_qc_dir, exist_ok=True)

masker = input_data.NiftiMasker(standardize=True, detrend=True, 
                                low_pass=None, high_pass=0.01, t_r=2.0,
                                smoothing_fwhm=6)

for sub in all_subs:
    # 1. Find all BOLD files for this subject/task
    bold_files = layout.get(subject=sub, task='Mult', suffix='bold', 
                            desc='preproc', extension='nii.gz')
    
    if not bold_files:
        print(f"⚠️  No BOLD found for sub-{sub}")
        continue

    # 2. Iterate through each BOLD file found (handles multiple runs/sessions)
    for bf in bold_files:
        entities = bf.get_entities()
        
        # 3. Find the EXACT matching confound file
        # We filter the entities to only keep the identifying ones
        search_entities = {k: v for k, v in entities.items() if k in ['subject', 'session', 'task', 'run']}
        
        cf = layout.get(suffix='confounds', extension='tsv', **search_entities)
        
        if not cf:
            print(f"⚠️  No matching confounds for sub-{sub} ses-{entities.get('session')} run-{entities.get('run')}")
            continue

        bold_path = bf.path
        confound_path = cf[0].path
        
        # Construct a unique ID for saving the plot
        run_id = f"sub-{sub}_ses-{entities.get('session', '01')}_run-{entities.get('run', '1')}"

        try:
            print(f"📊 Processing {run_id}...")
            
            # Load and Filter Confounds
            confounds_df = pd.read_csv(confound_path, sep='\t')
            cols = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z', 'global_signal', 'dvars']
            existing_cols = [c for c in cols if c in confounds_df.columns]
            selected_confounds = confounds_df[existing_cols].fillna(0)

            # Mask and Clean
            cleaned_data_array = masker.fit_transform(bold_path, confounds=selected_confounds)
            cleaned_img = masker.inverse_transform(cleaned_data_array)

            # tSNR Calculation
            tsnr_img = image.math_img("np.mean(img, axis=-1) / (np.std(img, axis=-1) + 1e-6)", img=cleaned_img)

            # Plotting
            fig = plt.figure(figsize=(10, 4))
            plotting.plot_epi(tsnr_img, title=f"{run_id} tSNR", display_mode='z', cut_coords=5, figure=fig)
            fig.savefig(os.path.join(output_qc_dir, f"{run_id}_tsnr.png"))
            plt.close(fig) 
            
            print(f"✅ Success: {run_id}")

        except Exception as e:
            print(f"❌ Error in {run_id}: {str(e)}")

Total BOLD files found: 240
⚠️  No matching confounds for sub-059 ses-T1 run-01
⚠️  No matching confounds for sub-059 ses-T1 run-02
⚠️  No matching confounds for sub-059 ses-T2 run-01
⚠️  No matching confounds for sub-059 ses-T2 run-02
⚠️  No matching confounds for sub-065 ses-T1 run-01
⚠️  No matching confounds for sub-065 ses-T1 run-02
⚠️  No matching confounds for sub-065 ses-T2 run-01
⚠️  No matching confounds for sub-065 ses-T2 run-02
⚠️  No matching confounds for sub-067 ses-T1 run-01
⚠️  No matching confounds for sub-067 ses-T1 run-02
⚠️  No matching confounds for sub-067 ses-T2 run-01
⚠️  No matching confounds for sub-067 ses-T2 run-02
⚠️  No matching confounds for sub-069 ses-T1 run-01
⚠️  No matching confounds for sub-069 ses-T1 run-02
⚠️  No matching confounds for sub-069 ses-T2 run-01
⚠️  No matching confounds for sub-069 ses-T2 run-02
⚠️  No matching confounds for sub-071 ses-T1 run-01
⚠️  No matching confounds for sub-071 ses-T1 run-02
⚠️  No matching confounds for sub-07